# Fine-tune EmbeddingGemma — Swiss Legal Retrieval
Dataset: `farwew/swiss-legal` (anchor, positive, negative triplets)

In [ ]:
# ⚠️ Run this cell FIRST, before any other cell — must be set before torch is imported
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
print('CUDA_VISIBLE_DEVICES =', os.environ['CUDA_VISIBLE_DEVICES'])
print('If you already ran other cells, restart the runtime and run from here.')

In [ ]:
!pip install -q -U sentence-transformers datasets
!pip install -q git+https://github.com/huggingface/transformers@v4.56.0-Embedding-Gemma-preview

In [ ]:
from huggingface_hub import login
login()  # masukkan HF token (butuh akses ke dataset private + model gemma)

In [ ]:
import torch
from sentence_transformers import SentenceTransformer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if torch.cuda.is_available():
    print('GPU :', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
    print('GPU count visible:', torch.cuda.device_count(), '(should be 1)')

model_id = 'google/embeddinggemma-300m'
model = SentenceTransformer(model_id, device=device)

model[0].auto_model.gradient_checkpointing_enable()
print('Gradient checkpointing enabled')
print('Max seq length:', model.max_seq_length)

In [ ]:
from datasets import load_dataset

ds = load_dataset('farwew/swiss-legal')
print(ds)
print('Columns:', ds['train'].column_names)
print('Train size:', len(ds['train']))

# Tampilkan sample
sample = ds['train'][0]
print('\nSample:')
print('  anchor  :', sample['anchor'][:100])
print('  positive:', sample['positive'][:100])
print('  negative:', sample['negative'][:100])

## Data Validation
Cek kualitas dataset sebelum training — null, duplikat, panjang teks, hard negative quality.

In [ ]:
import numpy as np

train_df = ds['train'].to_pandas()
split_name = 'eval' if 'eval' in ds else ('validation' if 'validation' in ds else None)
eval_df = ds[split_name].to_pandas() if split_name else None

print('=' * 55)
print('DATA VALIDATION REPORT')
print('=' * 55)

required_cols = ['anchor', 'positive', 'negative']
missing = [c for c in required_cols if c not in train_df.columns]
assert not missing, f'Missing columns: {missing}'
print(f'[OK] Required columns: {required_cols}')

for col in required_cols:
    n_null  = train_df[col].isna().sum()
    n_empty = (train_df[col].fillna('').str.strip() == '').sum()
    status = 'OK' if n_null == 0 and n_empty == 0 else 'WARN'
    print(f'[{status}] {col}: {n_null} null, {n_empty} empty')

n_dup_rows   = train_df.duplicated(subset=required_cols).sum()
n_anc_eq_pos = (train_df['anchor'] == train_df['positive']).sum()
n_pos_eq_neg = (train_df['positive'] == train_df['negative']).sum()
print(f'[{"OK" if n_dup_rows == 0 else "WARN"}] Duplicate triplets: {n_dup_rows}')
print(f'[{"OK" if n_anc_eq_pos == 0 else "WARN"}] anchor == positive: {n_anc_eq_pos}')
print(f'[{"OK" if n_pos_eq_neg == 0 else "WARN"}] positive == negative: {n_pos_eq_neg}')

print('\nText length (chars):')
for col in required_cols:
    lens = train_df[col].str.len()
    print(f'  {col:8s}: min={lens.min():<5} median={lens.median():<8.0f} max={lens.max():<6} mean={lens.mean():.0f}')

print('\nApprox token count (whitespace split):')
for col in required_cols:
    toks = train_df[col].str.split().str.len()
    pct95 = int(np.percentile(toks, 95))
    print(f'  {col:8s}: median={toks.median():.0f}  95th={pct95}  max={toks.max()}  (model max={model.max_seq_length})')
    if pct95 > model.max_seq_length:
        print(f'  [WARN] >5% of {col} may be truncated')

if eval_df is not None:
    print(f'\nEval split ({split_name}): {len(eval_df)} rows')
    print(f'  Rows with any null: {eval_df[required_cols].isna().any(axis=1).sum()}')
else:
    print('\n[WARN] No eval/validation split found')

print('\nHard negative quality check (200 train samples):')
n_check = min(200, len(train_df))
sub = ds['train'].select(range(n_check))
a_emb = model.encode(list(sub['anchor']),   prompt_name='Retrieval-query',    normalize_embeddings=True, show_progress_bar=False)
p_emb = model.encode(list(sub['positive']), prompt_name='Retrieval-document', normalize_embeddings=True, show_progress_bar=False)
n_emb = model.encode(list(sub['negative']), prompt_name='Retrieval-document', normalize_embeddings=True, show_progress_bar=False)
pos_sim = (a_emb * p_emb).sum(axis=1)
neg_sim = (a_emb * n_emb).sum(axis=1)
n_correct = (pos_sim > neg_sim).sum()
print(f'  Positive > Negative : {n_correct}/{n_check} ({100*n_correct/n_check:.1f}%)')
print(f'  Avg positive sim    : {pos_sim.mean():.3f}')
print(f'  Avg negative sim    : {neg_sim.mean():.3f}')
print(f'  Avg margin (pos-neg): {(pos_sim - neg_sim).mean():.3f}')
if n_correct / n_check < 0.5:
    print('  [WARN] <50% correct — negatives mungkin terlalu hard/mislabeled')
elif n_correct / n_check > 0.95:
    print('  [INFO] >95% correct — negatives terlalu easy')
else:
    print('  [OK] Good difficulty range')

print('\n' + '=' * 55)
print('Validation complete. Proceed to training.')
print('=' * 55)

In [ ]:
import numpy as np

samples = ds['train'].select(range(100))
a_emb = model.encode(list(samples['anchor']),   prompt_name='Retrieval-query',    normalize_embeddings=True)
p_emb = model.encode(list(samples['positive']), prompt_name='Retrieval-document', normalize_embeddings=True)
n_emb = model.encode(list(samples['negative']), prompt_name='Retrieval-document', normalize_embeddings=True)

pos_sim = (a_emb * p_emb).sum(axis=1)
neg_sim = (a_emb * n_emb).sum(axis=1)

print('=== SEBELUM TRAINING ===')
print(f'Positive > Negative: {(pos_sim > neg_sim).sum()}/100')
print(f'Avg positive sim   : {pos_sim.mean():.3f}')
print(f'Avg negative sim   : {neg_sim.mean():.3f}')

In [ ]:
from sentence_transformers import SentenceTransformerTrainer, SentenceTransformerTrainingArguments
from sentence_transformers.losses import MultipleNegativesRankingLoss

train_dataset = ds['train'].select_columns(['anchor', 'positive', 'negative'])
eval_dataset  = ds['eval'].select_columns(['anchor', 'positive', 'negative']) if 'eval' in ds else None

loss = MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir='finetuned-embeddinggemma-swiss-legal',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    learning_rate=1e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    lr_scheduler_type='cosine',
    fp16=torch.cuda.is_available(),
    prompts={
        'anchor':   model.prompts['Retrieval-query'],
        'positive': model.prompts['Retrieval-document'],
        'negative': model.prompts['Retrieval-document'],
    },
    logging_steps=50,
    report_to='none',
    save_strategy='epoch',
    save_total_limit=1,
)

trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    loss=loss,
)
trainer.train()

In [ ]:
a_emb = model.encode(list(samples['anchor']),   prompt_name='Retrieval-query',    normalize_embeddings=True)
p_emb = model.encode(list(samples['positive']), prompt_name='Retrieval-document', normalize_embeddings=True)
n_emb = model.encode(list(samples['negative']), prompt_name='Retrieval-document', normalize_embeddings=True)

pos_sim = (a_emb * p_emb).sum(axis=1)
neg_sim = (a_emb * n_emb).sum(axis=1)

print('=== SETELAH TRAINING ===')
print(f'Positive > Negative: {(pos_sim > neg_sim).sum()}/100')
print(f'Avg positive sim   : {pos_sim.mean():.3f}')
print(f'Avg negative sim   : {neg_sim.mean():.3f}')

## Evaluation on val.csv (Macro F1)

Evaluasi menggunakan metrik kompetisi: **Macro F1** per query.
- Query: `val.csv` (10 English queries dengan gold citations)
- Corpus: `laws_de.csv` + `court_considerations.csv`
- Format gold: `Art. 11 Abs. 2 OR;BGE 139 I 2 E. 3.1` (semicolon-separated)

In [ ]:
import os
import pandas as pd
import numpy as np

# Paths — sesuaikan jika beda
VAL_CSV   = 'data/val.csv'
LAWS_CSV  = 'data/laws_de.csv'
COURT_CSV = 'data/court_considerations.csv'

assert os.path.exists(VAL_CSV),  f'Not found: {VAL_CSV}'
assert os.path.exists(LAWS_CSV), f'Not found: {LAWS_CSV}'

# Load val queries
val_df = pd.read_csv(VAL_CSV)
print(f'Val queries: {len(val_df)}')
print(val_df[['query_id', 'query', 'gold_citations']].to_string(index=False))

# Load corpus
parts = []
laws_df = pd.read_csv(LAWS_CSV)[['citation', 'text']].dropna()
parts.append(laws_df)
print(f'\nlaws_de.csv: {len(laws_df):,} docs')

if os.path.exists(COURT_CSV):
    court_df = pd.read_csv(COURT_CSV)[['citation', 'text']].dropna()
    parts.append(court_df)
    print(f'court_considerations.csv: {len(court_df):,} docs')
else:
    print('[WARN] court_considerations.csv not found — skipping court corpus')

corpus_df = pd.concat(parts, ignore_index=True).drop_duplicates('citation')
citation_to_text = dict(zip(corpus_df['citation'], corpus_df['text']))
print(f'Total corpus: {len(citation_to_text):,} unique citations')

In [ ]:
# Build small corpus: hanya dokumen yang muncul di train positives + val gold citations
# Jauh lebih cepat dari encode 170k+ docs

# Reverse lookup: text → citation (dari laws_de + court)
text_to_cit = {v: k for k, v in citation_to_text.items()}

small_corpus = {}

# 1. Val gold citations (wajib ada di corpus supaya F1 bisa > 0)
for _, row in val_df.iterrows():
    if pd.notna(row.get('gold_citations')):
        for cit in str(row['gold_citations']).split(';'):
            cit = cit.strip()
            if cit and cit in citation_to_text:
                small_corpus[cit] = citation_to_text[cit]

print(f'Dari val gold citations: {len(small_corpus)} docs')

# 2. Train positives → cari citation-nya via reverse lookup
n_found = 0
for text in set(ds['train']['positive']):
    cit = text_to_cit.get(text)
    if cit and cit not in small_corpus:
        small_corpus[cit] = text
        n_found += 1

print(f'Dari train positives   : {n_found} docs tambahan')
print(f'Total small corpus     : {len(small_corpus)} docs')
print('(vs full corpus 170k+ docs — eval ini lebih cepat tapi lebih mudah karena tanpa distractor)')

# Encode corpus kecil
print('\nEncoding corpus...')
corpus_ids   = list(small_corpus.keys())
corpus_texts = list(small_corpus.values())

corpus_embs = model.encode(
    corpus_texts,
    prompt_name='Retrieval-document',
    normalize_embeddings=True,
    batch_size=256,
    show_progress_bar=True,
)
print(f'Corpus embeddings: {corpus_embs.shape}')

# Encode val queries
print('Encoding val queries...')
query_ids   = val_df['query_id'].tolist()
query_texts = val_df['query'].tolist()

query_embs = model.encode(
    query_texts,
    prompt_name='Retrieval-query',
    normalize_embeddings=True,
    show_progress_bar=False,
)

# Parse gold citations
gold_map = {}
for _, row in val_df.iterrows():
    if pd.notna(row.get('gold_citations')):
        gold_map[row['query_id']] = set(c.strip() for c in str(row['gold_citations']).split(';') if c.strip())
    else:
        gold_map[row['query_id']] = set()

In [ ]:
# Compute Macro F1 at different K values
scores = query_embs @ corpus_embs.T  # (n_queries, n_corpus)

def macro_f1_at_k(scores, query_ids, corpus_ids, gold_map, k):
    top_idx = np.argsort(scores, axis=1)[:, ::-1][:, :k]
    f1s = []
    for i, qid in enumerate(query_ids):
        pred = {corpus_ids[j] for j in top_idx[i]}
        gold = gold_map.get(qid, set())
        if not pred and not gold:
            f1s.append(1.0)
        elif not pred or not gold:
            f1s.append(0.0)
        else:
            tp = len(pred & gold)
            p  = tp / len(pred)
            r  = tp / len(gold)
            f1s.append(2 * p * r / (p + r) if (p + r) > 0 else 0.0)
    return float(np.mean(f1s)), f1s

print('=' * 55)
print('MACRO F1 EVALUATION (val.csv)')
print('=' * 55)
for k in [5, 10, 20, 50]:
    macro_f1, per_query = macro_f1_at_k(scores, query_ids, corpus_ids, gold_map, k)
    print(f'  Macro F1 @ {k:2d}: {macro_f1:.4f}')

# Per-query breakdown at K=10
print('\nPer-query breakdown @ K=10:')
_, per_query = macro_f1_at_k(scores, query_ids, corpus_ids, gold_map, 10)
top10_idx = np.argsort(scores, axis=1)[:, ::-1][:, :10]
for i, qid in enumerate(query_ids):
    pred = [corpus_ids[j] for j in top10_idx[i]]
    gold = gold_map.get(qid, set())
    n_gold = len(gold)
    n_correct = len(set(pred) & gold)
    print(f'  {qid}: F1={per_query[i]:.3f}  gold={n_gold}  correct={n_correct}/10  query={query_texts[i][:60]}')

## Reranking (BGE Cross-Encoder)

Flow: bi-encoder ambil **top-50** kandidat → cross-encoder re-score tiap pasangan `(query, doc)` → filter by threshold → compare F1.

In [ ]:
from sentence_transformers import CrossEncoder

# Download BGE reranker dari HuggingFace kalau belum ada
RERANKER_PATH = 'finetune/models/bge-reranker-v2-m3'

if not os.path.exists(RERANKER_PATH):
    from huggingface_hub import snapshot_download
    print('Downloading BGE reranker...')
    snapshot_download(
        repo_id='BAAI/bge-reranker-v2-m3',
        local_dir=RERANKER_PATH,
        ignore_patterns=['*.msgpack', '*.h5', 'flax_model*', 'tf_model*'],
    )
    print('Done.')

reranker = CrossEncoder(RERANKER_PATH, max_length=512, device=device)
print('Reranker loaded:', RERANKER_PATH)

In [ ]:
RERANK_TOP_N = 50   # ambil top-50 dari bi-encoder, lalu rerank

bi_scores = query_embs @ corpus_embs.T  # (n_queries, n_corpus)

# Rerank tiap query
reranked_preds = {}   # qid -> list of (reranker_score, citation)
for i, qid in enumerate(query_ids):
    # Step 1: bi-encoder top-50
    top50_idx = np.argsort(bi_scores[i])[::-1][:RERANK_TOP_N]
    candidates = [(corpus_ids[j], corpus_texts[j]) for j in top50_idx]

    # Step 2: cross-encoder re-score
    pairs = [(query_texts[i], doc_text) for _, doc_text in candidates]
    ce_scores = reranker.predict(pairs, batch_size=32, show_progress_bar=False)

    reranked_preds[qid] = sorted(
        zip(ce_scores, [cit for cit, _ in candidates]),
        reverse=True
    )

print('Reranking done.')
print(f'Sample reranked scores for {query_ids[0]}:')
for score, cit in reranked_preds[query_ids[0]][:5]:
    print(f'  {score:7.3f}  {cit}')

In [ ]:
# Compare: Bi-encoder vs Reranked — sweep threshold reranker (raw logit)
# BGE reranker output adalah raw logit (bukan 0-1), biasanya range -10 s/d +10

def f1_score(pred, gold):
    if not pred and not gold: return 1.0
    if not pred or not gold:  return 0.0
    tp = len(pred & gold)
    p, r = tp / len(pred), tp / len(gold)
    return 2 * p * r / (p + r) if (p + r) > 0 else 0.0

# Bi-encoder F1 @ K=10 (baseline)
bi_preds_k10 = {}
top10 = np.argsort(bi_scores, axis=1)[:, ::-1][:, :10]
for i, qid in enumerate(query_ids):
    bi_preds_k10[qid] = {corpus_ids[j] for j in top10[i]}
bi_f1 = np.mean([f1_score(bi_preds_k10[qid], gold_map.get(qid, set())) for qid in query_ids])

print(f'Bi-encoder Macro F1 @ K=10 : {bi_f1:.4f}')
print()

# Reranker threshold sweep
print(f'{"Threshold":>10}  {"Macro F1":>10}  {"Avg cit":>8}')
print('-' * 32)

best_f1, best_thresh = 0.0, 0.0
for thresh in [round(t, 1) for t in np.arange(-6, 8, 0.5)]:
    f1s = []
    for qid in query_ids:
        pred = {cit for score, cit in reranked_preds[qid] if score >= thresh}
        if not pred:  # fallback: ambil top-1
            pred = {reranked_preds[qid][0][1]}
        f1s.append(f1_score(pred, gold_map.get(qid, set())))
    macro_f1 = float(np.mean(f1s))
    avg_cit   = np.mean([len({c for s, c in reranked_preds[qid] if s >= thresh} or {reranked_preds[qid][0][1]}) for qid in query_ids])
    marker = ' <-- best' if macro_f1 > best_f1 else ''
    if macro_f1 > best_f1:
        best_f1, best_thresh = macro_f1, thresh
    print(f'{thresh:>10.1f}  {macro_f1:>10.4f}  {avg_cit:>8.1f}{marker}')

print(f'\nBest reranker threshold: {best_thresh:.1f}  (Macro F1 = {best_f1:.4f})')
print()
print('=' * 45)
print(f'  Bi-encoder  Macro F1: {bi_f1:.4f}')
print(f'  Reranked    Macro F1: {best_f1:.4f}  (threshold={best_thresh:.1f})')
winner = "Reranked" if best_f1 > bi_f1 else "Bi-encoder"
print(f'  Winner: {winner} (+{abs(best_f1 - bi_f1):.4f})')
print('=' * 45)

In [ ]:
# Simpan model
model.save('finetuned-embeddinggemma-swiss-legal/final')
print('Model saved.')

# Optional: push ke HuggingFace Hub
# model.push_to_hub('farwew/embeddinggemma-swiss-legal')

## Inference on test.csv → submission.csv

Flow:
1. Load **full corpus** (laws_de + court)
2. Encode dengan fine-tuned Gemma → bi-encoder top-50
3. BGE reranker re-score → filter threshold dari val sweep
4. Output `submission.csv`

In [ ]:
import os
import pandas as pd
import numpy as np

TEST_CSV  = 'data/test.csv'
LAWS_CSV  = 'data/laws_de.csv'
COURT_CSV = 'data/court_considerations.csv'

assert os.path.exists(TEST_CSV),  f'Not found: {TEST_CSV}'
assert os.path.exists(LAWS_CSV),  f'Not found: {LAWS_CSV}'

# Load test queries
test_df = pd.read_csv(TEST_CSV)
print(f'Test queries: {len(test_df)}')
print(test_df.head())

# Load full corpus
parts = [pd.read_csv(LAWS_CSV)[['citation', 'text']].dropna()]
print(f'\nlaws_de.csv: {len(parts[0]):,} docs')

if os.path.exists(COURT_CSV):
    court = pd.read_csv(COURT_CSV)[['citation', 'text']].dropna()
    parts.append(court)
    print(f'court_considerations.csv: {len(court):,} docs')
else:
    print('[WARN] court_considerations.csv tidak ditemukan — skip')

full_df = pd.concat(parts, ignore_index=True).drop_duplicates('citation')
test_corpus_ids   = full_df['citation'].tolist()
test_corpus_texts = full_df['text'].tolist()
print(f'Total corpus: {len(test_corpus_ids):,} docs')

In [ ]:
# Encode full corpus dengan fine-tuned Gemma (bisa 10-20 menit)
print('Encoding full corpus...')
test_corpus_embs = model.encode(
    test_corpus_texts,
    prompt_name='Retrieval-document',
    normalize_embeddings=True,
    batch_size=256,
    show_progress_bar=True,
)
print(f'Corpus embeddings: {test_corpus_embs.shape}')

# Encode test queries
print('Encoding test queries...')
test_query_embs = model.encode(
    test_df['query'].tolist(),
    prompt_name='Retrieval-query',
    normalize_embeddings=True,
    show_progress_bar=False,
)
print(f'Query embeddings: {test_query_embs.shape}')

In [ ]:
# Bi-encoder top-50 → BGE reranker → submission.csv
# Pakai best_thresh dari val sweep di atas (default -5.5 jika belum di-sweep)
RERANKER_THRESHOLD = best_thresh if 'best_thresh' in dir() else -5.5
RERANK_TOP_N = 50

print(f'Reranker threshold: {RERANKER_THRESHOLD}  (dari val sweep)')

bi_scores_test = test_query_embs @ test_corpus_embs.T  # (n_test, n_corpus)

rows = []
for i, qid in enumerate(test_df['query_id']):
    # Step 1: bi-encoder top-50
    top50_idx  = np.argsort(bi_scores_test[i])[::-1][:RERANK_TOP_N]
    candidates = [(test_corpus_ids[j], test_corpus_texts[j]) for j in top50_idx]

    # Step 2: BGE cross-encoder re-score
    pairs     = [(test_df['query'].iloc[i], doc_text) for _, doc_text in candidates]
    ce_scores = reranker.predict(pairs, batch_size=32, show_progress_bar=False)

    # Step 3: filter by threshold, fallback top-1 jika semua di bawah
    selected = [cit for score, (cit, _) in sorted(zip(ce_scores, candidates), reverse=True)
                if score >= RERANKER_THRESHOLD]
    if not selected:
        selected = [candidates[int(np.argmax(ce_scores))][0]]

    rows.append({'query_id': qid, 'predicted_citations': ';'.join(selected)})

submission = pd.DataFrame(rows)
submission.to_csv('submission.csv', index=False)

print(f'\nsubmission.csv saved — {len(submission)} rows')
print(submission.head(5).to_string(index=False))
cit_counts = submission['predicted_citations'].str.count(';') + 1
print(f'\nCitations per query: min={cit_counts.min()}  max={cit_counts.max()}  mean={cit_counts.mean():.1f}')

In [ ]:
# Download submission.csv
from google.colab import files
files.download('submission.csv')